# 01 — Exploratory Data Analysis

Explore the Brazilian E-Commerce (Olist) dataset — sales trends, seasonality, category patterns.

**Goal:** Understand data structure and uncover demand patterns to guide feature engineering.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_all_raw_data, join_tables, aggregate_daily_category
from src.config import RAW_DATA_DIR, CATEGORY_COL, TARGET_COL

sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 12})
%matplotlib inline

print('Libraries loaded successfully.')

## 1. Load Raw Data

In [ ]:
raw = load_all_raw_data()

for key, df in raw.items():
    print(f'{key:25s} → {len(df):>8,} rows × {len(df.columns)} cols')

## 2. Join & Clean

In [ ]:
unified = join_tables(raw)
print(f'Unified table: {len(unified):,} rows × {len(unified.columns)} cols')
unified.head(3)

In [ ]:
# Check missing values
missing = unified.isnull().sum()
missing[missing > 0].sort_values(ascending=False).head(20)

## 3. Daily Aggregation

In [ ]:
daily = aggregate_daily_category(unified)
print(f'Daily data: {len(daily):,} rows, {daily[CATEGORY_COL].nunique()} categories')
daily.head()

## 4. Time Series Overview

In [ ]:
# Overall daily orders
daily_total = daily.groupby('date')[TARGET_COL].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily_total['date'], daily_total[TARGET_COL], alpha=0.6, linewidth=0.8, label='Daily')
ax.plot(daily_total['date'], daily_total[TARGET_COL].rolling(14).mean(), 
        linewidth=2, color='red', label='14-day MA')
ax.set_title('Total Daily Orders (All Categories)')
ax.set_xlabel('Date')
ax.set_ylabel('Orders')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Total orders: {daily_total[TARGET_COL].sum():,}')
print(f'Avg daily: {daily_total[TARGET_COL].mean():.1f}')

## 5. Category Analysis

In [ ]:
cat_totals = daily.groupby(CATEGORY_COL)[TARGET_COL].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(range(len(cat_totals.head(20))), cat_totals.head(20).values[::-1], color='steelblue')
ax.set_yticks(range(20))
ax.set_yticklabels(cat_totals.head(20).index[::-1])
ax.set_title('Top 20 Categories by Total Orders')
ax.set_xlabel('Total Orders')
plt.tight_layout()
plt.show()

In [ ]:
# Category growth trends (top 5)
top5 = cat_totals.head(5).index.tolist()

fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
for i, cat in enumerate(top5):
    cat_data = daily[daily[CATEGORY_COL] == cat].sort_values('date')
    axes[i].plot(cat_data['date'], cat_data[TARGET_COL], linewidth=0.8, alpha=0.6)
    axes[i].plot(cat_data['date'], cat_data[TARGET_COL].rolling(7).mean(), linewidth=1.5, color='red')
    axes[i].set_ylabel('Orders')
    axes[i].set_title(cat)

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

## 6. Seasonality Patterns

In [ ]:
# Day-of-week pattern for top categories
daily_dow = daily.copy()
daily_dow['dow'] = pd.to_datetime(daily_dow['date']).dt.dayofweek
daily_dow['dow_name'] = daily_dow['dow'].map({
    0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'
})

fig, ax = plt.subplots(figsize=(10, 4))
dow_avg = daily_dow.groupby('dow_name')[TARGET_COL].mean()
dow_avg = dow_avg.reindex(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
ax.bar(dow_avg.index, dow_avg.values, color='steelblue')
ax.set_title('Average Orders by Day of Week')
ax.set_ylabel('Avg Orders')
plt.tight_layout()
plt.show()

In [ ]:
# Monthly pattern
daily_mth = daily.copy()
daily_mth['month'] = pd.to_datetime(daily_mth['date']).dt.month

fig, ax = plt.subplots(figsize=(10, 4))
month_avg = daily_mth.groupby('month')[TARGET_COL].mean()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
ax.bar(month_names, month_avg.values, color='steelblue')
ax.set_title('Average Orders by Month')
ax.set_ylabel('Avg Orders')
plt.tight_layout()
plt.show()

## 7. Revenue & Price Analysis

In [ ]:
if 'total_revenue' in daily.columns:
    revenue = daily.groupby(CATEGORY_COL)['total_revenue'].sum().sort_values(ascending=False).head(15)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(range(len(revenue)), revenue.values[::-1], color='green')
    ax.set_yticks(range(len(revenue)))
    ax.set_yticklabels(revenue.index[::-1])
    ax.set_title('Top 15 Categories by Revenue')
    ax.set_xlabel('Revenue (R$)')
    plt.tight_layout()
    plt.show()

In [ ]:
# Review score vs order volume
if 'avg_review_score' in daily.columns and 'total_revenue' in daily.columns:
    cat_agg = daily.groupby(CATEGORY_COL).agg({
        TARGET_COL: 'sum',
        'avg_review_score': 'mean',
        'total_revenue': 'sum',
    }).reset_index()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].scatter(cat_agg['avg_review_score'], cat_agg[TARGET_COL], alpha=0.6)
    axes[0].set_xlabel('Avg Review Score')
    axes[0].set_ylabel('Total Orders')
    axes[0].set_title('Review Score vs Order Volume')
    
    axes[1].scatter(cat_agg[TARGET_COL], cat_agg['total_revenue'], alpha=0.6, color='green')
    axes[1].set_xlabel('Total Orders')
    axes[1].set_ylabel('Total Revenue')
    axes[1].set_title('Orders vs Revenue')
    
    plt.tight_layout()
    plt.show()

## 8. Outlier Detection

In [ ]:
from src.config import TARGET_COL

# Detect outliers using IQR per category
def detect_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return (series < lower) | (series > upper)

daily['is_outlier'] = daily.groupby(CATEGORY_COL)[TARGET_COL].transform(detect_outliers)
outliers = daily[daily['is_outlier']]
print(f'Outliers detected: {len(outliers):,} ({len(outliers)/len(daily)*100:.1f}%)')

if len(outliers) > 0:
    outlier_cats = outliers.groupby(CATEGORY_COL).size().sort_values(ascending=False).head(10)
    print(f'\nTop categories with outliers:')
    for cat, count in outlier_cats.items():
        print(f'  {cat}: {count}')

## 9. Summary & Next Steps

In [ ]:
print('EDA Summary:')
print(f'  Dataset period: {daily["date"].min()} → {daily["date"].max()}')
print(f'  Total categories: {daily[CATEGORY_COL].nunique()}')
print(f'  Total orders: {daily[TARGET_COL].sum():,}')
print(f'  Avg daily orders: {daily.groupby("date")[TARGET_COL].sum().mean():.0f}')
print(f'  Peak daily orders: {daily.groupby("date")[TARGET_COL].sum().max():,}')
print()
print('Next: Feature Engineering')